# Lecture 6 — Class Exercise
## Choropleth Maps

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [2]:
# Task 1 — Life expectancy relative to the 2007 global average

# Load the Gapminder data and keep the 2007 observations
gm_2007 = px.data.gapminder().query("year == 2007").copy()

# Calculate the unweighted mean across countries and each country's deviation
global_mean_life_exp = gm_2007["lifeExp"].mean()
gm_2007["lifeExp_deviation"] = gm_2007["lifeExp"] - global_mean_life_exp

# Identify the continent with the lowest average life expectancy
continent_means = gm_2007.groupby("continent")["lifeExp"].mean().sort_values()
lowest_continent = continent_means.index[0]
lowest_continent_mean = continent_means.iloc[0]

print(f"Global mean life expectancy (2007): {global_mean_life_exp:.2f} years")
print("Mean life expectancy by continent:")
display(continent_means.rename("Mean life expectancy (years)").to_frame())

# A diverging scale is appropriate because values can be above or below zero.
# The midpoint is zero, which represents the global country-average life expectancy.
fig_task1 = px.choropleth(
    gm_2007,
    locations="iso_alpha",
    color="lifeExp_deviation",
    hover_name="country",
    hover_data={
        "country": False,
        "lifeExp": ":.1f",
        "lifeExp_deviation": ":+.2f",
        "iso_alpha": False,
    },
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    labels={
        "lifeExp_deviation": "Difference from global mean (years)",
        "lifeExp": "Life expectancy (years)",
    },
    title=(
        f"{lowest_continent} has the lowest average life expectancy "
        f"({lowest_continent_mean:.1f} years) in 2007"
    ),
)

fig_task1.update_layout(
    title_x=0.5,
    coloraxis_colorbar_title="Years from<br>global mean",
    margin=dict(l=10, r=10, t=80, b=10),
)
fig_task1.show()


Global mean life expectancy (2007): 67.01 years
Mean life expectancy by continent:


,Mean life expectancy (years)
continent,
Africa,54.806038
Asia,70.728485
Americas,73.608120
Europe,77.648600
Oceania,80.719500


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** [datasets/geo-countries — Country Polygons as GeoJSON](https://github.com/datasets/geo-countries), using the raw file at `https://raw.githubusercontent.com/datasets/geo-countries/main/data/countries.geojson`. The boundaries are based on Natural Earth.

**Chart type chosen:** `px.choropleth`. It draws country polygons directly from the GeoJSON and works well for a static global comparison. It does not require a Mapbox token.

**GeoJSON feature key:** `properties.name`. The GeoJSON dataset documents a `name` property for each country/territory. The code inspects the first feature's properties before using this key.

**Colour scale chosen:** Sequential (`Viridis`). HDI is a non-negative index where higher values indicate higher human development; there is no meaningful zero-centred positive/negative split, so a sequential scale is appropriate.

**Insight title:** The chart title is generated from the data and names the region with the lowest mean HDI among the regions represented in the supplied CSV.

**Data matching:** The CSV's `Country` column is matched to the GeoJSON `properties.name` field. A small alias dictionary handles common naming differences. The code prints any unmatched countries so they can be checked rather than silently assigning incorrect boundaries.


In [ ]:
# Task 2 — HDI by country using the supplied world_development.csv

# Load the provided dataset (keep the CSV in the same folder as this notebook)
development = pd.read_csv("world_development.csv")

# Download the public country-boundary GeoJSON from the documented source.
# The file is saved locally so it can be reused without downloading it again.
import os
import requests

geojson_url = (
    "https://raw.githubusercontent.com/datasets/geo-countries/"
    "main/data/countries.geojson"
)
geojson_path = "countries.geojson"

if not os.path.exists(geojson_path):
    response = requests.get(geojson_url, timeout=60)
    response.raise_for_status()
    with open(geojson_path, "w", encoding="utf-8") as geojson_file:
        geojson_file.write(response.text)

with open(geojson_path, "r", encoding="utf-8") as geojson_file:
    world_geojson = json.load(geojson_file)

# Inspect the GeoJSON properties before choosing featureidkey.
print("Example GeoJSON feature properties:")
print(world_geojson["features"][0]["properties"])
feature_id_key = "properties.name"

# Match country names between the CSV and GeoJSON.
# Keys are names in the CSV; values are names used by the GeoJSON.
country_aliases = {
    "United States": "United States of America",
    "DR Congo": "Democratic Republic of the Congo",
}

development["geojson_country"] = development["Country"].replace(country_aliases)

# Check which names are available in the GeoJSON before plotting.
geojson_names = {
    feature["properties"]["name"]
    for feature in world_geojson["features"]
    if "name" in feature.get("properties", {})
}
matched = development[development["geojson_country"].isin(geojson_names)].copy()
unmatched_countries = sorted(
    set(development["Country"]) - set(matched["Country"])
)

print(f"Countries in CSV: {len(development)}")
print(f"Countries matched to GeoJSON: {len(matched)}")
if unmatched_countries:
    print("Countries not matched (check spelling/aliases):", unmatched_countries)

# Find the region with the lowest mean HDI in this supplied dataset.
regional_hdi = (
    development.groupby("Region", as_index=False)["HDI"]
    .mean()
    .sort_values("HDI")
)
lowest_hdi_region = regional_hdi.iloc[0]["Region"]
lowest_hdi_value = regional_hdi.iloc[0]["HDI"]

print("\nMean HDI by region:")
display(regional_hdi.sort_values("HDI", ascending=False))

# px.choropleth uses the GeoJSON country polygons. A sequential scale is
# suitable because HDI is an index whose magnitude runs from low to high.
fig_task2 = px.choropleth(
    matched,
    geojson=world_geojson,
    locations="geojson_country",
    featureidkey=feature_id_key,
    color="HDI",
    hover_name="Country",
    hover_data={
        "Region": True,
        "HDI": ":.3f",
        "Life_Expectancy": ":.1f",
        "GNI_per_capita": ":,",
        "geojson_country": False,
    },
    color_continuous_scale="Viridis",
    labels={
        "HDI": "Human Development Index",
        "Life_Expectancy": "Life expectancy (years)",
        "GNI_per_capita": "GNI per capita",
    },
    title=(
        f"{lowest_hdi_region} has the lowest mean HDI "
        f"({lowest_hdi_value:.3f}) in this dataset"
    ),
)

fig_task2.update_geos(
    showcountries=True,
    showcoastlines=True,
    showframe=False,
    projection_type="natural earth",
)
fig_task2.update_layout(
    title_x=0.5,
    coloraxis_colorbar_title="HDI",
    margin=dict(l=10, r=10, t=80, b=10),
)
fig_task2.show()


Example GeoJSON feature properties:
{'name': 'Indonesia', 'ISO3166-1-Alpha-3': 'IDN', 'ISO3166-1-Alpha-2': 'ID'}
Countries in CSV: 20
Countries matched to GeoJSON: 20

Mean HDI by region:


,Region,HDI
2,Europe,0.937200
4,North America,0.921000
1,East Asia,0.870667
3,Latin America,0.756000
0,Africa,0.587200
5,South Asia,0.579000
